# 17 Joins

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Combine DataFrames with every join type (inner, left, right, full, cross, semi, anti), write join conditions correctly, avoid duplicate-column and row-explosion problems, and explain how Spark executes a join (broadcast hash join vs sort-merge join).
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Joins are where pipelines connect facts to dimensions, enrich events, and detect new or deleted records. They are also the most common cause of slow jobs (huge shuffles, skew) and of wrong numbers (duplicated rows, lost rows). Expect several interview questions on join types <b>and</b> join strategies. This lesson is <b>interview-critical</b>.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Join types</b><br>
| Type | Keeps | Typical use |<br>|---|---|---|<br>| <code>inner</code> | Only rows with a match on both sides | Facts with valid dimension keys |<br>| <code>left</code> (<code>left_outer</code>) | All left rows, nulls where no match | Enrich without losing facts |<br>| <code>right</code> (<code>right_outer</code>) | All right rows | Rare. Swap sides and use left |<br>| <code>full</code> (<code>full_outer</code>) | All rows from both sides | Reconciliation between two systems |<br>| <code>left_semi</code> | Left rows that <b>have</b> a match, left columns only | <code>WHERE EXISTS</code> / filter by another table |<br>| <code>left_anti</code> | Left rows with <b>no</b> match, left columns only | New records, orphans, <code>NOT EXISTS</code> |<br>| <code>cross</code> | Every combination (rows × rows) | Small grids, date × store scaffolds |
</div>

```
   customers (left)          orders (right)
   C1 Amal                   O1 -> C1
   C2 John                   O2 -> C1
   C3 Vivek                  O3 -> C2
                             O4 -> C9 (unknown customer)

inner      : C1-O1, C1-O2, C2-O3
left       : C1-O1, C1-O2, C2-O3, C3-null
right      : C1-O1, C1-O2, C2-O3, null-O4
full       : C1-O1, C1-O2, C2-O3, C3-null, null-O4
left_semi  : C1, C2            (customers who ordered)
left_anti  : C3                (customers who never ordered)
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A bank enriches 50 million daily card transactions with a 2,000-row merchant table and a 5-million-row customer table. The merchant join should be a <b>broadcast join</b>: the small table is copied to every executor, so the 50 million transactions don't move. The customer join is large on both sides, so Spark uses a <b>sort-merge join</b>, which shuffles both sides by <code>customer_id</code>. A nightly <code>left_anti</code> join of today's customer list against yesterday's finds the newly opened accounts.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df1.join(df2, "id")                                   # equi-join on a common column name (one id column kept)
df1.join(df2, ["id", "date"], "left")                 # several keys
df1.join(df2, df1.cust_id == df2.id, "inner")         # different names (both columns kept)
df1.join(df2, (df1.id == df2.id) & (df2.active), "left")  # extra conditions
df1.join(F.broadcast(df2), "id")                      # broadcast hint
df1.join(df2, "id", "left_anti")                      # rows in df1 not in df2
df1.crossJoin(df2)
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates customers, orders, and a small country dimension. The data includes an order for an unknown customer, a customer with no orders, and a duplicate key in the dimension.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 customers, 6 orders (one for unknown customer <code>C9</code>), and 4 country rows where <code>IN</code> appears twice on purpose.
</div>

In [0]:
from pyspark.sql import functions as F

customers = spark.createDataFrame(
    [("C1", "Amal", "IN"), ("C2", "John", "UK"), ("C3", "Vivek", "AE"), ("C4", "Sara", None)],
    "customer_id STRING, name STRING, country STRING",
)
orders = spark.createDataFrame(
    [
        ("O1", "C1", 120.0, "2024-07-01"),
        ("O2", "C1", 80.0, "2024-07-03"),
        ("O3", "C2", 200.0, "2024-07-03"),
        ("O4", "C9", 55.0, "2024-07-04"),
        ("O5", "C2", 15.0, "2024-07-05"),
        ("O6", "C3", 300.0, "2024-07-06"),
    ],
    "order_id STRING, customer_id STRING, amount DOUBLE, order_date STRING",
).withColumn("order_date", F.to_date("order_date"))
countries = spark.createDataFrame(
    [("IN", "India"), ("IN", "Bharat"), ("UK", "United Kingdom"), ("AE", "United Arab Emirates")],
    "country STRING, country_name STRING",
)
customers.show(); orders.show(); countries.show()

## 1. Inner join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins orders to customers on <code>customer_id</code>.<br><br>
<b>Why it matters</b><br>Only matching rows survive. Order <code>O4</code> (unknown customer) and customer <code>C4</code> (no orders) both disappear. That is correct for some questions and a silent data loss for others.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
5 rows: O1, O2, O3, O5, O6, each with the customer's name. A single <code>customer_id</code> column, because the key was given as a string.
</div>

In [0]:
inner = orders.join(customers, "customer_id", "inner")
inner.orderBy("order_id").show()
print("orders:", orders.count(), "| after inner join:", inner.count())

## 2. Left, right and full outer joins

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs the three outer joins and compares row counts.<br><br>
<b>Why it matters</b><br><code>left</code> keeps every row of the main (left) table, which is the safest way to enrich facts. <code>full</code> is used to reconcile two sources and see what exists only on one side.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>left</code> (orders → customers): 6 rows, O4 with a null name. <code>right</code>: 6 rows, with C4 and no order. <code>full</code>: 7 rows, including both O4 and C4.
</div>

In [0]:
left = orders.join(customers, "customer_id", "left")
right = orders.join(customers, "customer_id", "right")
full = orders.join(customers, "customer_id", "full")

print("left:", left.count(), "| right:", right.count(), "| full:", full.count())
full.orderBy("customer_id", "order_id").show()

## 3. Semi and anti joins

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Finds customers who have ordered (<code>left_semi</code>) and customers who never ordered (<code>left_anti</code>), plus orders with no valid customer.<br><br>
<b>Why it matters</b><br>Semi and anti joins return <b>only the left columns</b> and never duplicate left rows, even if there are many matches. They are the cleanest way to write <code>EXISTS</code> and <code>NOT EXISTS</code>, and anti joins are the standard tool for finding new, missing or orphan records.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Semi: C1, C2 and C3 (once each, even though C1 and C2 have 2 orders). Anti: C4. Orphan orders: O4.
</div>

In [0]:
print("Customers with at least one order:")
customers.join(orders, "customer_id", "left_semi").show()

print("Customers with no orders:")
customers.join(orders, "customer_id", "left_anti").show()

print("Orders whose customer doesn't exist (orphans):")
orders.join(customers, "customer_id", "left_anti").show()

## 4. Different key names and duplicate columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins when the key columns have different names, shows the duplicate columns that result, and fixes them.<br><br>
<b>Why it matters</b><br>Joining on a <b>string</b> column name keeps one key column. Joining on an <b>expression</b> keeps both, which leads to <code>AMBIGUOUS_REFERENCE</code> errors later. Fix with <code>drop</code>, aliases, or by renaming before the join.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The expression join has both <code>cust_id</code> and <code>customer_id</code>. The cleaned version keeps only one.
</div>

In [0]:
orders_renamed = orders.withColumnRenamed("customer_id", "cust_id")

expr_join = orders_renamed.join(customers, orders_renamed.cust_id == customers.customer_id, "inner")
print(expr_join.columns)

# Fix 1: drop the redundant column via the DataFrame reference
print(expr_join.drop(customers.customer_id).columns)

# Fix 2: DataFrame aliases and an explicit select
o, c = orders_renamed.alias("o"), customers.alias("c")
o.join(c, F.col("o.cust_id") == F.col("c.customer_id")) \
 .select("o.order_id", F.col("c.customer_id"), "c.name", "o.amount").orderBy("order_id").show()

## 5. Edge case: duplicate keys multiply rows

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins customers to <code>countries</code>, where <code>IN</code> appears twice.<br><br>
<b>Why it matters</b><br>If the right side has N rows for a key, every matching left row is repeated N times. That <b>silently inflates totals</b>, which is one of the most common join bugs in production. Always check that dimension keys are unique before joining, or deduplicate first.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 customers become 5 rows: Amal appears twice (India and Bharat). The uniqueness check reports <code>IN</code> with 2 rows. After deduplicating, the join returns 4 rows again.
</div>

In [0]:
exploded = customers.join(countries, "country", "left")
print("customers:", customers.count(), "| after join:", exploded.count())
exploded.orderBy("customer_id").show()

# Defensive check: are the dimension keys unique?
countries.groupBy("country").count().filter("count > 1").show()

# Fix: make the dimension unique on the join key first (here: keep the first name alphabetically)
countries_unique = countries.groupBy("country").agg(F.min("country_name").alias("country_name"))
print("after dedup join:", customers.join(countries_unique, "country", "left").count())

## 6. Multiple keys, extra conditions and non-equi joins

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins on two keys, adds a filter inside the join condition, and does a range (non-equi) join to assign each order to a price band.<br><br>
<b>Why it matters</b><br>For a <b>left</b> join, a condition inside the join is not the same as a filter after it. Inside the join, non-matching left rows are kept with nulls. In a <code>WHERE</code> afterwards, they are removed. Range joins (<code>between</code>) are common for bands, validity periods and SCD Type 2 lookups.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The condition-in-join version keeps all 6 orders (null promo where it doesn't apply). The filter-after version keeps only the matching ones. The band join labels each order <code>small</code>, <code>medium</code> or <code>large</code>.
</div>

In [0]:
promos = spark.createDataFrame(
    [("C1", "2024-07-03", "10OFF"), ("C2", "2024-07-03", "FREESHIP"), ("C3", "2024-07-01", "OLD")],
    "customer_id STRING, promo_date STRING, promo STRING",
).withColumn("promo_date", F.to_date("promo_date"))

# Two keys with different names on the date side
cond = (orders.customer_id == promos.customer_id) & (orders.order_date == promos.promo_date)
print("Condition inside a LEFT join (all orders kept):")
orders.join(promos, cond, "left").select(orders["*"], "promo").orderBy("order_id").show()

print("Same join followed by a filter (unmatched orders removed):")
orders.join(promos, cond, "left").filter(F.col("promo").isNotNull()).select(orders["*"], "promo").show()

# Non-equi (range) join
bands = spark.createDataFrame([("small", 0.0, 100.0), ("medium", 100.0, 250.0), ("large", 250.0, 1e9)],
                              "band STRING, low DOUBLE, high DOUBLE")
orders.join(bands, (orders.amount >= bands.low) & (orders.amount < bands.high), "left") \
      .select("order_id", "amount", "band").orderBy("order_id").show()

## 7. Cross join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds every combination of stores and dates, which is a scaffold for "fill in the days with zero sales".<br><br>
<b>Why it matters</b><br>Cross joins are legitimate for small scaffolds but explode quickly (rows × rows). Spark requires you to ask for one explicitly with <code>crossJoin</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 stores × 3 dates = 9 rows. Joining the actual sales onto the scaffold with a left join shows 0 for days without sales.
</div>

In [0]:
stores = spark.createDataFrame([("S1",), ("S2",), ("S3",)], "store STRING")
dates = spark.sql("SELECT explode(sequence(date'2024-07-01', date'2024-07-03', interval 1 day)) AS day")
scaffold = stores.crossJoin(dates)
print("scaffold rows:", scaffold.count())

daily_sales = spark.createDataFrame([("S1", "2024-07-01", 500.0), ("S2", "2024-07-03", 120.0)],
                                    "store STRING, day STRING, sales DOUBLE").withColumn("day", F.to_date("day"))
scaffold.join(daily_sales, ["store", "day"], "left").na.fill({"sales": 0.0}).orderBy("store", "day").show()

## 8. Joins in SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes inner, left and anti joins in Spark SQL, including the <code>NOT EXISTS</code> and <code>LEFT ANTI JOIN</code> forms.<br><br>
<b>Why it matters</b><br>The exam and Databricks SQL use SQL syntax. <code>LEFT SEMI JOIN</code> and <code>LEFT ANTI JOIN</code> are Spark SQL extensions you may not know from other databases.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Revenue per customer name (C4 with 0), and C4 twice as the customer with no orders.
</div>

In [0]:
customers.createOrReplaceTempView("customers_v")
orders.createOrReplaceTempView("orders_v")

spark.sql("""
    SELECT c.customer_id, c.name, coalesce(sum(o.amount), 0) AS revenue
    FROM customers_v c
    LEFT JOIN orders_v o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY c.customer_id
""").show()

spark.sql("SELECT * FROM customers_v c LEFT ANTI JOIN orders_v o ON c.customer_id = o.customer_id").show()
spark.sql("""
    SELECT * FROM customers_v c
    WHERE NOT EXISTS (SELECT 1 FROM orders_v o WHERE o.customer_id = c.customer_id)
""").show()

## Under the hood: join strategies

```
BROADCAST HASH JOIN  (one side small)
  driver collects the small table -> sends a copy to every executor
  each executor builds a hash table and streams its partitions of the big table through it
  -> NO shuffle of the big table. Fastest option when one side fits in memory.

SORT-MERGE JOIN  (both sides large, equi-join)  -- Spark's default for big joins
  Exchange hashpartitioning(key) on BOTH sides      <- shuffle
  Sort each partition by key
  Merge the two sorted streams
  -> scales to any size, but pays two shuffles and two sorts

SHUFFLED HASH JOIN
  shuffle both sides, build a hash table from the smaller side per partition (no sort)

BROADCAST NESTED LOOP / CARTESIAN
  used for non-equi joins and cross joins -> compares many pairs, can be very slow
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> sort-merge and shuffled hash joins shuffle both sides by the join key.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> a broadcast hash join doesn't shuffle the large side.

Spark picks a broadcast join automatically when a side is smaller than `spark.sql.autoBroadcastJoinThreshold` (10 MB by default), and **AQE** can switch to a broadcast at runtime once it sees the real sizes. You can force a strategy with hints: `F.broadcast(df)`, `df.hint("broadcast")`, `df.hint("merge")`, `df.hint("shuffle_hash")`.

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The first plan shows <code>BroadcastHashJoin</code> with a <code>BroadcastExchange</code>. The second shows <code>SortMergeJoin</code> with an <code>Exchange hashpartitioning</code> and a <code>Sort</code> on each side.
</div>

In [0]:
big = spark.range(2_000_000).withColumn("key", F.col("id") % 1000)
small = spark.range(1000).withColumnRenamed("id", "key").withColumn("label", F.concat(F.lit("k"), F.col("key")))

print("---- broadcast hint ----")
big.join(F.broadcast(small), "key").explain()

print("---- sort-merge hint ----")
big.join(small.hint("merge"), "key").explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: row count goes up after a join</b><br>
The right side has duplicate keys. Check uniqueness with <code>groupBy(key).count().filter("count &gt; 1")</code> and deduplicate the dimension.<br><br>
<b>⛔ Problem: rows silently disappear</b><br>An inner join dropped unmatched keys, or the keys are null, or there is a type or format mismatch (<code>"001"</code> vs <code>1</code>, trailing spaces, different case). Compare with a <code>left_anti</code> join to see what didn't match.<br><br>
**⛔ Problem: <code>[AMBIGUOUS_REFERENCE]</code> after a join**<br>Both sides have a column with the same name. Join on a string key, drop one side's column, or use aliases.<br><br>
<b>⛔ Problem: a left join behaves like an inner join</b><br>A <code>filter</code> on a right-side column after the join removed the null rows. Move that condition into the join condition.<br><br>
<b>⛔ Problem: join is extremely slow or never finishes</b><br>Accidental cross product (missing or always-true condition), a non-equi join on large tables, or data skew on a hot key. Check the plan for <code>CartesianProduct</code> / <code>BroadcastNestedLoopJoin</code>, and see <code>05_optimization</code>.<br><br>
<b>⛔ Problem: broadcast join fails with out-of-memory</b><br>The "small" side isn't small. Remove the hint and let Spark use sort-merge, or filter and select fewer columns first.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Explain the different join types in Spark.</b><br>
Inner keeps matches only. Left, right and full outer keep all rows from the left, right or both sides with nulls where there is no match. Left semi returns left rows that have a match, with left columns only. Left anti returns left rows with no match. Cross returns every combination.<br><br>

<b>🎤 2. What is the difference between a left semi join and an inner join?</b><br>
A semi join returns only the left table's columns, and each left row appears at most once no matter how many matches it has. An inner join returns columns from both sides and repeats left rows for every match.<br><br>

<b>🎤 3. How does a broadcast hash join work and when is it used?</b><br>
The smaller table is collected and copied to every executor, where it is put in a hash table. The large table is then joined partition by partition without being shuffled. Spark uses it automatically when a side is below <code>autoBroadcastJoinThreshold</code> (10 MB by default), when AQE detects a small side at runtime, or when you add a broadcast hint.<br><br>

<b>🎤 4. How does a sort-merge join work?</b><br>
Both sides are shuffled by the join key so matching keys end up in the same partition. Each partition is sorted by key, and then the two sorted streams are merged. It is the default for large equi-joins because it scales and can spill to disk.<br><br>

<b>🎤 5. Why might a join return more rows than the left table has?</b><br>
Because the right side has duplicate keys. Each left row is repeated once per matching right row. Deduplicate or aggregate the right side first.<br><br>

<b>🎤 6. Why do rows disappear even with matching IDs?</b><br>
Common causes are null keys (null never equals null), type differences, leading zeros, whitespace or case differences. A left anti join shows exactly which keys failed to match.<br><br>

<b>🎤 7. What is the difference between putting a condition in the ON clause and in WHERE for a left join?</b><br>
In the ON clause, it only affects which right rows match, and all left rows are kept. In WHERE, it runs after the join and removes left rows whose right side is null, which turns the left join into an inner join.<br><br>

<b>🎤 8. How do you find records that exist in one table but not in another?</b><br>
A left anti join, or <code>NOT EXISTS</code> in SQL. Avoid <code>NOT IN</code> when the subquery can contain nulls.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer needs all rows from <code>transactions</code> that have NO matching <code>account_id</code> in <code>accounts</code>, returning only the columns of <code>transactions</code>. Which join type should they use?</b><br>
A. <code>left</code><br>
B. <code>left_semi</code><br>
C. <code>left_anti</code><br>
D. <code>full</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> An anti join returns left rows without a match, and only the left columns.</details><br><br>

<b>Q2. A 500 GB fact table is joined to a 5 MB lookup table. What will Spark most likely do, and why?</b><br>
A. Sort-merge join, because it is the default<br>
B. Broadcast hash join, because the lookup table is below the broadcast threshold<br>
C. Cartesian product, because the sizes differ<br>
D. Shuffle both tables to the driver<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Small tables under the threshold (10 MB by default) are broadcast, which avoids shuffling the large table.</details><br><br>

<b>Q3. <code>customers</code> has 100 rows with unique <code>id</code>. <code>addresses</code> has 150 rows, and some customers have two addresses. How many rows can <code>customers INNER JOIN addresses ON customers.id = addresses.customer_id</code> return?</b><br>
A. Exactly 100<br>
B. At most 100<br>
C. Up to 150<br>
D. Exactly 250<br>
<details><summary><b>Show answer</b></summary><b>C.</b> Each address matches one customer, so the result has at most one row per address. Customers with two addresses appear twice.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>customers</code>, <code>orders</code> and <code>countries</code>:
<ol><li>Total revenue per customer <b>name</b>, including customers with no orders (show 0)</li><li>List orders whose customer is unknown, and customers who never ordered, in two anti joins</li><li>Join customers to countries <b>without</b> duplicating Amal, keeping <code>India</code> as the name</li><li>For each order, show the customer's name and country name, keeping all orders</li><li>Explain in a comment why <code>orders.join(customers, "customer_id", "left").filter(customers.country == "IN")</code> behaves like an inner join, and rewrite it so all orders are kept</li><li>Force a broadcast join of <code>orders</code> with <code>customers</code> and confirm it in the plan</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Revenue per customer including those without orders
customers.join(orders, "customer_id", "left").groupBy("name").agg(
    F.coalesce(F.sum("amount"), F.lit(0.0)).alias("revenue")
).orderBy("name").show()

# 2. Two anti joins
orders.join(customers, "customer_id", "left_anti").show()
customers.join(orders, "customer_id", "left_anti").show()

# 3. Unique dimension first: prefer 'India' for IN
countries_fixed = countries.filter(~((F.col("country") == "IN") & (F.col("country_name") != "India")))
customers.join(countries_fixed, "country", "left").show()

# 4. Keep all orders
orders.join(customers, "customer_id", "left").join(countries_fixed, "country", "left") \
      .select("order_id", "name", "country_name", "amount").orderBy("order_id").show()

# 5. The filter runs after the join and removes rows where country is null (unmatched orders),
#    so only matched rows survive. Put the condition in the join instead:
c_in = customers.filter(F.col("country") == "IN")
orders.join(c_in, "customer_id", "left").orderBy("order_id").show()

# 6. Broadcast hint
orders.join(F.broadcast(customers), "customer_id").explain()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Inner, left, right, full, semi, anti, cross. Semi/anti return left columns only and never duplicate left rows</li><li>Join on a string key name to keep one key column. Expression joins keep both</li><li>Duplicate keys on the right multiply rows. Check uniqueness of dimension keys</li><li>Null keys never match with <code>=</code></li><li>For left joins, conditions in ON keep left rows. Filters in WHERE remove them</li><li>Broadcast hash join: small side copied to executors, no shuffle of the big side</li><li>Sort-merge join: both sides shuffled and sorted, the default for large equi-joins</li><li>AQE can switch to broadcast at runtime. Hints: <code>broadcast</code>, <code>merge</code>, <code>shuffle_hash</code></li></ul>
</div>

| Task | Code |
|---|---|
| Same key name | `a.join(b, "id", "left")` |
| Several keys | `a.join(b, ["id", "dt"])` |
| Different names | `a.join(b, a.cid == b.id)` |
| Exists | `a.join(b, "id", "left_semi")` |
| Not exists | `a.join(b, "id", "left_anti")` |
| Broadcast | `a.join(F.broadcast(b), "id")` |
| Cross | `a.crossJoin(b)` |
| Check key uniqueness | `b.groupBy("id").count().filter("count > 1")` |

## Git commit

```
git add 01_spark_basics/17_joins.ipynb
git commit -m "add 17 joins notebook"
```